In [1]:
import warnings
# Ignore all warnings
warnings.filterwarnings('ignore')

# Enable interactive mode with a scrollable plot
%matplotlib qt
import numpy as np
import glob
import matplotlib.pyplot as plt
import sys
import pickle as pkl
sys.path.append("..")

import mne
from mne.io import RawArray
from mne.channels import make_standard_montage
from mne.preprocessing import read_ica
from mne.io import read_raw_fif
import gc
import fooof
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_rel

# import mne_microstates
import pandas as pd
import seaborn as sns
import matplotlib.patches as mpatches

import os
from scipy.io import loadmat
import scipy

import matplotlib.image as mpimg
from matplotlib.colors import ListedColormap
import pyxdf
import socket
import pickle as pkl
from pyprep.find_noisy_channels import NoisyChannels
import copy
from pathlib import Path
import nibabel as nib
import json
import nibabel as nib
from nilearn.image import new_img_like
from nilearn.plotting import plot_img_on_surf
from matplotlib.colors import ListedColormap

%load_ext autoreload
%autoreload 2

C:\Users\specsdcn\AppData\Local\Temp\ipykernel_19556\2444545934.py:20: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  import fooof


In [2]:
device_name = socket.gethostname()
print(device_name)

DESKTOP-I0FNO1F


In [3]:
if device_name == "LAPTOP-LRI6535I":
    path_analysis_data = "E:/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    plots_path = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/analysis_plots/"

if device_name == "kritrim":
    path_analysis_data = "C:/Users/manda/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"

if device_name == "DESKTOP-I0FNO1F":
    path_analysis_data = "D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    plots_path = "D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/"



In [4]:
from helpers.helper_functions import get_matched_legths, plot_significance_single_sub_band, plot_all_subs_single_band, get_fooof_aperiodic, build_vector_grid_pdf, plot_relative_difference_single_sub_band, get_fooof_bands
from helpers.topo_eeg import get_fdr_p_values,  get_band_power_difference_all_subjects_significance_aperiodic_fdr, get_channel_wise_fooof
from helpers.source_level_psd_helpers import *
from helpers.group_level_helpers import group_level_bootstrap_test, latex_group_vs_individual_table
from helpers.frequency_analysis import compute_psd_mne


def store_mne_raw(eeg_filtered, path_folder, path_data, ovwrite=False):
    eeg_filtered.save(path_folder + path_data, overwrite=ovwrite)

def load_mne_raw(path_folder, path_data):
    return mne.io.read_raw_fif(path_folder + path_data, preload=True)



Python Executable Path: 
c:\Users\specsdcn\anaconda3\envs\pdoc\python.exe


In [5]:
import warnings
warnings.filterwarnings('ignore')

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}

pat_state_dict = {
    'sub01': 'UWS',
    'sub03': 'MCS+',
    'sub04': 'MCS-',
    'sub07': 'MCS-',
    'sub09': 'MCS-',
    'sub10': 'MCS-',
    'sub12': 'MCS-',
}

sfreq = 512
paradigms = {0: "noise", 1: "music"}
pat_state = {"sub01": "UWS", "sub03": "MCS+", "sub04": "MCS-", "sub07":"MCS-", "sub09":"MCS-", "sub10":"MCS-", "sub12":"MCS-"}


#### Load AAL Atlas

In [6]:
aal_dir = path_analysis_data_new + "eeg_source/"

with open(f'{aal_dir}AAL_labels.json', 'r') as f:
    aal_labels = json.load(f)

In [7]:
aal_labels

['Precentral_L',
 'Precentral_R',
 'Frontal_Sup_L',
 'Frontal_Sup_R',
 'Frontal_Sup_Orb_L',
 'Frontal_Sup_Orb_R',
 'Frontal_Mid_L',
 'Frontal_Mid_R',
 'Frontal_Mid_Orb_L',
 'Frontal_Mid_Orb_R',
 'Frontal_Inf_Oper_L',
 'Frontal_Inf_Oper_R',
 'Frontal_Inf_Tri_L',
 'Frontal_Inf_Tri_R',
 'Frontal_Inf_Orb_L',
 'Frontal_Inf_Orb_R',
 'Rolandic_Oper_L',
 'Rolandic_Oper_R',
 'Supp_Motor_Area_L',
 'Supp_Motor_Area_R',
 'Olfactory_L',
 'Olfactory_R',
 'Frontal_Sup_Medial_L',
 'Frontal_Sup_Medial_R',
 'Frontal_Med_Orb_L',
 'Frontal_Med_Orb_R',
 'Rectus_L',
 'Rectus_R',
 'Insula_L',
 'Insula_R',
 'Cingulum_Ant_L',
 'Cingulum_Ant_R',
 'Cingulum_Mid_L',
 'Cingulum_Mid_R',
 'Cingulum_Post_L',
 'Cingulum_Post_R',
 'Hippocampus_L',
 'Hippocampus_R',
 'ParaHippocampal_L',
 'ParaHippocampal_R',
 'Amygdala_L',
 'Amygdala_R',
 'Calcarine_L',
 'Calcarine_R',
 'Cuneus_L',
 'Cuneus_R',
 'Lingual_L',
 'Lingual_R',
 'Occipital_Sup_L',
 'Occipital_Sup_R',
 'Occipital_Mid_L',
 'Occipital_Mid_R',
 'Occipital_Inf_L'

#### Load Source Reconstructed Data

In [8]:
with open(path_analysis_data_new + "all_eeg_source_reconstruction_eloreta_45_adhoccov.pkl", "rb") as f:
    eeg_source_reconstructed = pkl.load(f)


In [9]:
eeg_source_reconstructed["sub01"]["rest_init"].shape

(80, 116, 2560)

In [10]:
eeg_source_reconstructed_cont = {}

for sub in eeg_source_reconstructed:
    eeg_source_reconstructed_cont[sub] = {}
    for parad in  eeg_source_reconstructed[sub]:
        eeg_source_reconstructed_cont[sub][parad] = eeg_source_reconstructed[sub][parad].transpose(1, 0, 2).reshape(116, -1)


In [11]:
a = np.empty_like(eeg_source_reconstructed_cont[sub][parad])
a.shape

(116, 171520)

In [12]:
count_cerebrum_labels = 0 

for label in aal_labels:

    if label.startswith("Cerebelum") or label.startswith("Vermis"):
        continue

    else:
        count_cerebrum_labels += 1

count_cerebrum_labels

90

In [13]:
eeg_source_reconstructed_cont_cerebrum = {}
eeg_source_reconstructed_cont_cerebrum_raw = {}

regions_all = eeg_source_reconstructed_cont[sub][parad].shape[0]

n_regions = 90
ch_types = ['eeg'] * n_regions

info = mne.create_info(ch_names=ch_types, sfreq=512, ch_types=ch_types)

for sub in eeg_source_reconstructed_cont:
    eeg_source_reconstructed_cont_cerebrum[sub] = {}
    eeg_source_reconstructed_cont_cerebrum_raw[sub] = {}

    for parad in  eeg_source_reconstructed_cont[sub]:
        eeg_source_reconstructed_cont_cerebrum[sub][parad] = np.zeros([count_cerebrum_labels, eeg_source_reconstructed_cont[sub][parad].shape[1]])

        for reg in range(regions_all):
            reg_label = aal_labels[reg]

            if reg_label.startswith("Cerebelum") or reg_label.startswith("Vermis"):
                continue

            else:
                eeg_source_reconstructed_cont_cerebrum[sub][parad][reg, :] = eeg_source_reconstructed_cont[sub][parad][reg, :]

        eeg_source_reconstructed_cont_cerebrum_raw[sub][parad] = mne.io.RawArray(eeg_source_reconstructed_cont_cerebrum[sub][parad], info, verbose=False)

In [14]:
eeg_source_reconstructed_cont_cerebrum_raw['sub07'].keys()

dict_keys(['rest_init', 'reward', 'pink_segs', 'music_segs', 'rest_mid', 'rest_end'])

FOOOF

In [15]:
band_powers_all = {sub: {parad: [] for parad in eeg_source_reconstructed_cont_cerebrum[sub]} for sub in eeg_source_reconstructed_cont_cerebrum}
psds_all = {sub: {parad: [] for parad in eeg_source_reconstructed_cont_cerebrum[sub]} for sub in eeg_source_reconstructed_cont_cerebrum}

subjects = list(eeg_source_reconstructed_cont_cerebrum.keys())
paradigms = list(eeg_source_reconstructed_cont_cerebrum[subjects[0]].keys())


f_min = bands["delta"][0]
f_max = bands["gamma low"][1]

segment_parads = ['pink_segs', 'music_segs']
power_type = "relative"

for sub in eeg_source_reconstructed_cont_cerebrum:
    for parad in  eeg_source_reconstructed_cont_cerebrum[sub]:

        # if parad in segment_parads:
        #     segment_secs = 3

        # else:
        segment_secs = 5
        
        segment_size = int(sfreq * segment_secs)
        overlap = int(segment_size /2)

        eeg_array = eeg_source_reconstructed_cont_cerebrum[sub][parad]
        n_regions = eeg_array.shape[0]
        ch_types = ['eeg'] * n_regions

        info = mne.create_info(ch_names=ch_types, sfreq=sfreq, ch_types=ch_types)
        eeg_raw = mne.io.RawArray(eeg_array, info, verbose=False)

        band_powers, psds, freqs = compute_psd_mne(eeg_raw, bands, overlap, segment_size, f_min, f_max, power_type=power_type)

        band_powers_all[sub][parad] = band_powers
        psds_all[sub][parad] = [psds, freqs]

Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective wind

In [16]:
# with open(path_analysis_data_new + "psds_all_eloreta_45_adhoccov.pkl", "wb") as f:
#     pkl.dump(psds_all, f)

In [17]:
# with open(path_analysis_data_new + "psds_all_eloreta_45_adhoccov.pkl", "rb") as f:
#     psds_all = pkl.load(f)

In [18]:
psds_all.keys()

dict_keys(['sub01', 'sub03', 'sub04', 'sub07', 'sub10', 'sub12', 'sub09'])

In [19]:
# eeg_fooof_all = get_fooof(psds_all, n_channels=90, save_path=path_analysis_data_new + "fooof_all_source/")

LOAD FOOOF

In [20]:
fooof_path = path_analysis_data_new + "fooof_all_sources_adhoccov/"
fooof_files = glob.glob(fooof_path + "*.pkl")


In [21]:
print(Path)

<class 'matplotlib.path.Path'>


In [22]:
from pathlib import Path as Path 

eeg_fooof_all_subs = {sub:{} for sub in eeg_source_reconstructed_cont_cerebrum}

for fl in fooof_files:
    file_stem = Path(fl).stem
    file_name = Path(fl).name

    sub_name = file_stem.split("_")[2][:5]
    parad_name = file_stem.split("_")[2][5:]

    try:
        parad_name_2 = file_stem.split("_")[3]
        parad_name += "_" + parad_name_2

    except:
        parad_name = parad_name
    
    print(sub_name, parad_name)
    
    with open(fl, "rb") as f:
        eeg_fooof_all_subs[sub_name][parad_name] = pkl.load(f)
        f.close()
        

sub01 rest_init
sub01 reward
sub03 rest_init
sub03 reward
sub04 rest_init
sub04 reward
sub07 music_segs
sub07 pink_segs
sub07 rest_end
sub07 rest_init
sub07 rest_mid
sub07 reward
sub09 music_segs
sub09 pink_noise
sub09 pink_segs
sub09 rest_end
sub09 rest_init
sub09 rest_mid
sub09 reward
sub10 music_segs
sub10 pink_noise
sub10 pink_segs
sub10 rest_init
sub10 rest_mid
sub10 reward
sub12 music_segs
sub12 pink_noise
sub12 pink_segs
sub12 rest_end
sub12 rest_init
sub12 rest_mid
sub12 reward


### Plot the Differences in FOOOF between Conditions

Import AAL Atlas

In [23]:
data_dir = path_analysis_data_new + "eeg_source/"

with open(f'{data_dir}AAL_labels.json', 'r') as f:
    aal_labels = json.load(f)

In [24]:
data_dir = path_analysis_data_new + "eeg_source/"
aal_nii_path = f'{data_dir}AAL.nii'   # adjust filename if needed
aal_img = nib.load(aal_nii_path)

aal_data = np.squeeze(np.asanyarray(aal_img.dataobj)).astype(int)
print("AAL Atlas Shape: ", aal_data.shape)

AAL Atlas Shape:  (91, 109, 91)


In [25]:
aal_regions = np.unique(aal_data)
aal_regions = aal_regions[aal_regions != 0]
aal_regions

data_dict = {
    "id": [],
    "regions": [],
    "labels": []
}

with open(path_analysis_data_new + "eeg_source/aal1_for_SPM8/ROI_MNI_V4.txt", "r") as f:
    for line in f:
        if line.strip():
            parts = line.split()  # split by whitespace
            
            data_dict["id"].append(parts[0])
            data_dict["regions"].append(parts[1])
            data_dict["labels"].append(parts[2])

print(data_dict.keys())


aal_regions_str = [str(reg) for reg in aal_regions]

count_reg_similar = 0

for reg in aal_regions_str:
    if reg in data_dict["labels"]:
        count_reg_similar += 1

print("Similar Regions Number: ", count_reg_similar)


label_region_mapping = {}

for indx in range(len(data_dict["labels"])):
    label_region_mapping[int(data_dict["labels"][indx])] = data_dict["regions"][indx]


dict_keys(['id', 'regions', 'labels'])
Similar Regions Number:  116


#### 1. Rest vs. Reward

In [26]:
eeg_fooof_rest_reward = {}


eeg_fooof_rest_reward["sub01"] = {}
eeg_fooof_rest_reward["sub01"]["rest_init"] = eeg_fooof_all_subs["sub01"]["rest_init"].copy()
eeg_fooof_rest_reward["sub01"]["reward"] = eeg_fooof_all_subs["sub01"]["reward"].copy()

eeg_fooof_rest_reward["sub03"] = {}
eeg_fooof_rest_reward["sub03"]["rest_init"] = eeg_fooof_all_subs["sub03"]["rest_init"].copy()
eeg_fooof_rest_reward["sub03"]["reward"] = eeg_fooof_all_subs["sub03"]["reward"].copy()

eeg_fooof_rest_reward["sub04"] = {}
eeg_fooof_rest_reward["sub04"]["rest_init"] = eeg_fooof_all_subs["sub04"]["rest_init"].copy()
eeg_fooof_rest_reward["sub04"]["reward"] = eeg_fooof_all_subs["sub04"]["reward"].copy()

eeg_fooof_rest_reward["sub07"] = {}
eeg_fooof_rest_reward["sub07"]["rest_init"] = eeg_fooof_all_subs["sub07"]["rest_init"].copy()
eeg_fooof_rest_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_reward["sub09"] = {} 
eeg_fooof_rest_reward["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_reward["sub10"] = {} 
eeg_fooof_rest_reward["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_rest_reward["sub12"] = {} 
eeg_fooof_rest_reward["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [27]:
eeg_fooof_rest_reward_matched = get_matched_legths(eeg_fooof_rest_reward)

Subject: sub01, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 159 epochs
(90, 223, 159) (90, 223, 159)
Subject: sub03, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 153 epochs
(90, 223, 153) (90, 223, 153)
Subject: sub04, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 69 epochs
(90, 223, 69) (90, 223, 69)
Subject: sub07, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 145 epochs
(90, 223, 145) (90, 223, 145)
Subject: sub09, Paradigms: ['rest_init', 'reward']
Truncated array1 from 175 → 133 epochs (dropped last 42)
(90, 223, 133) (90, 223, 133)
Subject: sub10, Paradigms: ['rest_init', 'reward']
Truncated array1 from 95 → 85 epochs (dropped last 10)
(90, 223, 85) (90, 223, 85)
Subject: sub12, Paradigms: ['rest_init', 'reward']
Truncated array1 from 139 → 105 epochs (dropped last 34)
(90, 223, 105) (90, 223, 105)


In [28]:
sub_id = "sub10"

freqs = psds_all[sub_id]["rest_init"][1]
psds = psds_all[sub_id]["rest_init"][0]

n_regions = 90

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}

_, rest_init_music_fooof_bands_pvalues_abs, fooof_rest_reward = get_channel_wise_fooof(freqs, eeg_fooof_rest_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals = get_fdr_p_values(rest_init_music_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)


sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
Paradigm 0 rest_init
Paradigm 1 reward
sub09
Paradigm 0 rest_init
Paradigm 1 reward
sub10
Paradigm 0 rest_init
Paradigm 1 reward
sub12
Paradigm 0 rest_init
Paradigm 1 reward


In [29]:
mcs_minus_subjects = ["sub04", "sub07", "sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_rest_reward,
    mcs_minus_subjects,
    condition1="rest_init",
    condition2="reward",   # confirm this matches your actual music-condition key
    bands=bands,
    abs_rel=0,
    n_boot=1000,
    n_channels=90
)


for band in bands:
    print(f"{band}: {sig_mask[band].sum()} / 128 channels significant at group level")


# First comparison — produces the full table opening
latex_code_1 = latex_group_vs_individual_table(
    fdr_corrected_p_vals, mcs_minus_subjects, sig_mask, bands,
    comparison_label="Rest vs. Continuous Music",
    mode='start',
    n_channels=90,
    caption="Group-level versus individual-level significance across source regions and frequency bands.",
    label="tab:group_vs_individual_source",
)


print(latex_code_1)

delta: 0 / 128 channels significant at group level
theta: 0 / 128 channels significant at group level
alpha: 0 / 128 channels significant at group level
beta: 0 / 128 channels significant at group level
gamma low: 0 / 128 channels significant at group level
\begin{table}[htbp]
\centering
\caption{Group-level versus individual-level significance across source regions and frequency bands.}
\label{tab:group_vs_individual_source}

\begin{tabularx}{\textwidth}{@{}l l X X X X@{}}
\toprule
\textbf{Comparison} &
\textbf{Band} &
\textbf{Group-level sig.} &
\textbf{Sig. in $\geq$1 subject} &
\textbf{Sig. in majority} &
\textbf{Sig. in all subjects} \\
\midrule

\multirow{5}{*}{\shortstack[l]{Rest vs.\\Continuous Music}}
& $\delta$ & 0/90 & 38/90 & 0/90 ($\geq$3/5) & 0/90 \\
& $\theta$ & 0/90 & 73/90 & 1/90 ($\geq$3/5) & 0/90 \\
& $\alpha$ & 0/90 & 79/90 & 24/90 ($\geq$3/5) & 0/90 \\
& $\beta$ & 0/90 & 88/90 & 18/90 ($\geq$3/5) & 0/90 \\
& $\gamma$ & 0/90 & 83/90 & 33/90 ($\geq$3/5) & 0/90 \\

\b

In [30]:
fdr_corrected_p_vals.keys()

dict_keys(['sub01', 'sub03', 'sub04', 'sub07', 'sub09', 'sub10', 'sub12'])

In [31]:
fdr_corrected_p_vals["sub04"]["alpha"] < 0.05

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False,  True, False, False, False, False, False, False,  True,
        True, False,  True, False,  True, False,  True, False,  True,
       False,  True, False,  True, False,  True, False,  True, False,
        True, False, False, False, False, False, False, False,  True,
       False,  True, False, False, False, False, False, False, False,
        True, False,  True, False,  True, False, False,  True, False,
        True, False, False,  True,  True, False, False,  True, False])

In [32]:
label_region_mapping

{2001: 'Precentral_L',
 2002: 'Precentral_R',
 2101: 'Frontal_Sup_L',
 2102: 'Frontal_Sup_R',
 2111: 'Frontal_Sup_Orb_L',
 2112: 'Frontal_Sup_Orb_R',
 2201: 'Frontal_Mid_L',
 2202: 'Frontal_Mid_R',
 2211: 'Frontal_Mid_Orb_L',
 2212: 'Frontal_Mid_Orb_R',
 2301: 'Frontal_Inf_Oper_L',
 2302: 'Frontal_Inf_Oper_R',
 2311: 'Frontal_Inf_Tri_L',
 2312: 'Frontal_Inf_Tri_R',
 2321: 'Frontal_Inf_Orb_L',
 2322: 'Frontal_Inf_Orb_R',
 2331: 'Rolandic_Oper_L',
 2332: 'Rolandic_Oper_R',
 2401: 'Supp_Motor_Area_L',
 2402: 'Supp_Motor_Area_R',
 2501: 'Olfactory_L',
 2502: 'Olfactory_R',
 2601: 'Frontal_Sup_Medial_L',
 2602: 'Frontal_Sup_Medial_R',
 2611: 'Frontal_Med_Orb_L',
 2612: 'Frontal_Med_Orb_R',
 2701: 'Rectus_L',
 2702: 'Rectus_R',
 3001: 'Insula_L',
 3002: 'Insula_R',
 4001: 'Cingulum_Ant_L',
 4002: 'Cingulum_Ant_R',
 4011: 'Cingulum_Mid_L',
 4012: 'Cingulum_Mid_R',
 4021: 'Cingulum_Post_L',
 4022: 'Cingulum_Post_R',
 4101: 'Hippocampus_L',
 4102: 'Hippocampus_R',
 4111: 'ParaHippocampal_L',
 4

In [33]:
_, eeg_fooof_bands_abs_rest_init_music = get_fooof_bands(eeg_fooof_rest_reward_matched, freqs, psds, bands)


In [34]:
subjects_sorted = sorted(fdr_corrected_p_vals.keys())
sub_label_dict = {sub: f"SUB{i+1}" for i, sub in enumerate(subjects_sorted)}
sub_label_dict

{'sub01': 'SUB1',
 'sub03': 'SUB2',
 'sub04': 'SUB3',
 'sub07': 'SUB4',
 'sub09': 'SUB5',
 'sub10': 'SUB6',
 'sub12': 'SUB7'}

In [35]:
eeg_fooof_bands_abs_rest_init_music["sub01"].keys()

dict_keys(['rest_init', 'reward'])

In [36]:
def plot_fooof_fdr_results(fdr_corrected_p_vals, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color,view, pat_state_dict, save_dir1, 
                           save_dir2, sub_label_dict=None, out_filename=None):
    
    for sub_id in fdr_corrected_p_vals.keys():
        for band in bands:
            plot_significance_single_sub_band(
                fdr_corrected_p_vals=fdr_corrected_p_vals,
                aal_img=aal_img,
                label_region_mapping=label_region_mapping,
                sub_id=sub_id,
                band=band,
                parad1=parad1,
                parad2=parad2,
                view=view,
                pvals=pvals,
                cmap_color=cmap_color,
                save_dir=save_dir1,
            )

    build_vector_grid_pdf(
        image_dir=save_dir1,
        parad1=parad1,
        parad2=parad2,
        bands=bands,  # controls row order
        pat_state=pat_state_dict,
        save_dir=save_dir2,
        sub_labels=sub_label_dict,
        out_filename=out_filename
    )




In [37]:
save_path1 = f"{plots_path}/fdr_source/rest_reward/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/rest_reward/fdr_plots/merged/"

parad1 = "Rest"
parad2 = "Music Cont."
view = "lateral"

# cmap_color = ListedColormap(['#DAE8EC', "#D45E00"])  # warm greige / muted amber
cmap_color = ListedColormap(["#EDF4F6", "#7DB17B"])   # non-sig pale / significant green, pencil-matchedpvals = True 

legend_label1 = "Rest"
legend_label2 = "Music Cont."
pvals=True
plot_fooof_fdr_results(fdr_corrected_p_vals, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, pat_state_dict, 
                       save_path1, save_path2, out_filename="src_sig_rest_musiccont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/fdr_plots/sub01_Rest_Music Cont._gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/fdr_plots/sub03_Rest_Music Cont._delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/mus

In [38]:
def plot_fooof_relative_difference_results(power_dict, bands, aal_img, label_region_mapping, condition1, condition2, cmap_color, view, pat_state_list, 
                                           save_dir1, save_dir2, sub_label_dict=None, out_filename=None):

    for sub_id in power_dict.keys():
        for band in bands:
            plot_relative_difference_single_sub_band(
                power_dict=power_dict,
                aal_img=aal_img,
                label_region_mapping=label_region_mapping,
                cmap_color=cmap_color,
                sub_id=sub_id,
                band=band,
                condition1=condition1,
                condition2=condition2,
                view=view,
                save_dir=save_dir1)

    build_vector_grid_pdf(
        image_dir=save_dir1,
        parad1=condition1,
        parad2=condition2,
        bands=bands,
        pat_state=pat_state_list,
        save_dir=save_dir2,
        sub_labels=sub_label_dict,
        out_filename=out_filename)





In [39]:

_, eeg_fooof_bands_abs_rest_init_music = get_fooof_bands(eeg_fooof_rest_reward_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_rest_init_music,
    bands, aal_img, label_region_mapping,
    condition1="rest_init", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/rest_reward/color/",
    save_dir2=f"{plots_path}/fdr_source/rest_reward/color/merged/",
    out_filename="src_reldiff_rest_musiccont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/color/sub01_rest_init_reward_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/color/sub01_rest_init_reward_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/color/sub01_rest_init_reward_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/color/sub01_rest_init_reward_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/color/sub01_rest_init_reward_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/color/sub03_rest_init_reward_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/pl

In [40]:
summary = build_cluster_summary_all(
    fdr_corrected_p_vals, eeg_fooof_bands_abs_rest_init_music,
    condition1="rest_init", condition2="reward", bands=bands,
    label_region_mapping=label_region_mapping,
    region_to_anatomical_lobe=region_to_anatomical_lobe,
)

plot_cluster_bubble_heatmap(
    summary, bands,
    cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
    pat_state=pat_state_dict, sub_labels=sub_label_dict,
    condition1_label="Rest", condition2_label="Music Cont.",
    save_path=f"{plots_path}/fdr_source/rest_reward/color/bubble/src_bubble_rest_musiccont.pdf",
)


Aperiodic

In [41]:
eeg_fooof_rest_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_reward_matched)
fdr_corrected_p_vals_rest_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_reward_aperiodic, pat_state, n_channels=n_regions)



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward


In [42]:
aperiodic_params = ["Exponent", "Offset"] 
save_path1 = f"{plots_path}/fdr_source/rest_reward/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/rest_reward/aperiodic/merged/"

parad1 = "rest"
parad2 = "reward"
view = "lateral"
# cmap_color = ListedColormap(['#DAE8EC', "#D45E00"])  # warm greige / muted amber
cmap_color = ListedColormap(["#EDF4F6", "#7DB17B"])   # non-sig pale / significant green, pencil-matched
pvals = True 

plot_fooof_fdr_results(fdr_corrected_p_vals_rest_reward_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, 
                       view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_rest_musiccont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic/sub01_rest_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic/sub01_rest_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic/sub03_rest_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic/sub03_rest_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic/sub04_rest_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic/sub04_rest_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis

In [43]:
bands = ["Exponent", "Offset"]  # Aperiodic parameters  
plot_fooof_relative_difference_results(
    eeg_fooof_rest_reward_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="rest_init", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/rest_reward/aperiodic_color/",
    save_dir2=f"{plots_path}/fdr_source/rest_reward/aperiodic_color/merged/",
    out_filename="src_aperiodic_reldiff_rest_musiccont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic_color/sub01_rest_init_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic_color/sub01_rest_init_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic_color/sub03_rest_init_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic_color/sub03_rest_init_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic_color/sub04_rest_init_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/aperiodic_color/sub04_rest_init_reward_Offset.pdf
Saved: D:/Taran/

#### 2. Music Segs vs. Pink Segs

In [44]:
eeg_fooof_all_subs["sub07"].keys()

dict_keys(['music_segs', 'pink_segs', 'rest_end', 'rest_init', 'rest_mid', 'reward'])

In [45]:
eeg_fooof_music_pink_segs = {}

eeg_fooof_music_pink_segs["sub07"] = {}
eeg_fooof_music_pink_segs["sub07"]["pink_segs"] = eeg_fooof_all_subs["sub07"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub07"]["music_segs"] = eeg_fooof_all_subs["sub07"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub09"] = {} 
eeg_fooof_music_pink_segs["sub09"]["pink_segs"] = eeg_fooof_all_subs["sub09"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub09"]["music_segs"] = eeg_fooof_all_subs["sub09"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub10"] = {} 
eeg_fooof_music_pink_segs["sub10"]["pink_segs"] = eeg_fooof_all_subs["sub10"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub10"]["music_segs"] = eeg_fooof_all_subs["sub10"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub12"] = {} 
eeg_fooof_music_pink_segs["sub12"]["pink_segs"] = eeg_fooof_all_subs["sub12"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub12"]["music_segs"] = eeg_fooof_all_subs["sub12"]["music_segs"].copy()

eeg_fooof_music_pink_segs_matched = get_matched_legths(eeg_fooof_music_pink_segs)

Subject: sub07, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 255 → 249 epochs (dropped last 6)
(90, 223, 249) (90, 223, 249)
Subject: sub09, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 187 → 185 epochs (dropped last 2)
(90, 223, 185) (90, 223, 185)
Subject: sub10, Paradigms: ['pink_segs', 'music_segs']
Truncated array2 from 135 → 69 epochs (dropped last 66)
(90, 223, 69) (90, 223, 69)
Subject: sub12, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 165 → 163 epochs (dropped last 2)
(90, 223, 163) (90, 223, 163)


In [46]:
bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}

In [47]:
sub_id = "sub10"

freqs = psds_all[sub_id]["rest_init"][1]
psds = psds_all[sub_id]["rest_init"][0]

n_regions = 90

_, music_pink_seg_fooof_bands_pvalues_abs, fooof_music_pink_segs = get_channel_wise_fooof(freqs, eeg_fooof_music_pink_segs_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_music_pink_segs = get_fdr_p_values(music_pink_seg_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)


sub07
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub09
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub10
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub12
Paradigm 0 pink_segs
Paradigm 1 music_segs


In [48]:
mcs_minus_subjects = ["sub07", "sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_music_pink_segs,
    mcs_minus_subjects,
    condition1="pink_segs",
    condition2="music_segs",   # confirm this matches your actual music-condition key
    bands=bands,
    abs_rel=0,
    n_boot=1000,
    n_channels=90

)

for band in bands:
    print(f"{band}: {sig_mask[band].sum()} / 128 channels significant at group level")


latex_code_2 = latex_group_vs_individual_table(
    fdr_corrected_p_vals_music_pink_segs, mcs_minus_subjects, sig_mask, bands,
    comparison_label="Segmented Music vs. Segmented Pink Noise",
    mode='append',
    n_channels=90
)

print(latex_code_2)




delta: 0 / 128 channels significant at group level
theta: 0 / 128 channels significant at group level
alpha: 0 / 128 channels significant at group level
beta: 0 / 128 channels significant at group level
gamma low: 0 / 128 channels significant at group level
\midrule

\multirow{5}{*}{\shortstack[l]{Segmented Music vs.\\Segmented Pink Noise}}
& $\delta$ & 0/90 & 0/90 & 0/90 ($\geq$2/4) & 0/90 \\
& $\theta$ & 0/90 & 0/90 & 0/90 ($\geq$2/4) & 0/90 \\
& $\alpha$ & 0/90 & 0/90 & 0/90 ($\geq$2/4) & 0/90 \\
& $\beta$ & 0/90 & 0/90 & 0/90 ($\geq$2/4) & 0/90 \\
& $\gamma$ & 0/90 & 0/90 & 0/90 ($\geq$2/4) & 0/90 \\




In [49]:
save_path1 = f"{plots_path}/fdr_source/music_pink_segs/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/music_pink_segs/fdr_plots/merged/"

parad1 = "Pink Noise Segs"
parad2 = "Music Segs"
view = "lateral"

# cmap_color = ListedColormap(['#DAE8EC', "#D45E00"])  # warm greige / muted amber
cmap_color = ListedColormap(["#EDF4F6", "#6FB46F"])   # non-sig pale / SIGNIFICANT green
pvals = True 
legend_label1 = "Pink Noise Segs"
legend_label2 = "Music Segs"

plot_fooof_fdr_results(fdr_corrected_p_vals_music_pink_segs, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, 
                       pat_state_dict, save_path1, save_path2, out_filename="src_sig_musicsegs_pinksegs.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/fdr_plots/sub07_Pink Noise Segs_Music Segs_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/fdr_plots/sub09_Pink Noise Seg

In [50]:
save_path1 = f"{plots_path}/fdr_source/music_pink_segs/color/"
save_path2 = f"{plots_path}/fdr_source/music_pink_segs/color/merged/"

# parad1 = "Music Segs"
# parad2 = "Pink Noise Segs"
view = "lateral"
# cmap_color = ListedColormap(['white', 'black'])
pvals = True 

_, eeg_fooof_bands_abs_pink_music = get_fooof_bands(eeg_fooof_music_pink_segs_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_pink_music,
    bands, aal_img, label_region_mapping,
    condition1="pink_segs", condition2="music_segs",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2, out_filename="src_reldiff_musicsegs_pinksegs.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/color/sub07_pink_segs_music_segs_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/color/sub09_pink_segs_music_segs_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Un

In [51]:
_, eeg_fooof_bands_abs_pink_music = get_fooof_bands(eeg_fooof_music_pink_segs_matched, freqs, psds, bands)

summary = build_cluster_summary_all(
    fdr_corrected_p_vals_music_pink_segs, eeg_fooof_bands_abs_pink_music,
    condition1="pink_segs", condition2="music_segs", bands=bands,
    label_region_mapping=label_region_mapping,
    region_to_anatomical_lobe=region_to_anatomical_lobe,
)

plot_cluster_bubble_heatmap(
    summary, bands,
    cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
    pat_state=pat_state_dict, sub_labels=sub_label_dict,
    condition1_label="Pink Noise Seg.", condition2_label="Music Seg.",
    save_path=f"{plots_path}/fdr_source/music_pink_segs/color/bubble/src_bubble_musicsegs_pinksegs.pdf",
)


Aperiodic

In [52]:
eeg_fooof_music_pink_segs_aperiodic = get_fooof_aperiodic(eeg_fooof_music_pink_segs_matched)
fdr_corrected_p_vals_music_pink_segs_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_music_pink_segs_aperiodic, pat_state, n_channels=n_regions)

Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs


In [53]:
aperiodic_params = ["Exponent", "Offset"] 
save_path1 = f"{plots_path}/fdr_source/music_pink_segs/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/music_pink_segs/aperiodic/merged/"

parad1 = "Pink Noise Segs"
parad2 = "Music Segs"
view = "lateral"
# cmap_color = ListedColormap(['#DAE8EC', "#D45E00"])  # warm greige / muted amber
cmap_color = ListedColormap(["#EDF4F6", "#6FB46F"])   # non-sig pale / SIGNIFICANT green
pvals = True 

plot_fooof_fdr_results(fdr_corrected_p_vals_music_pink_segs_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, 
                       cmap_color, view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_musicsegs_pinksegs.pdf")  



Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic/sub07_Pink Noise Segs_Music Segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic/sub07_Pink Noise Segs_Music Segs_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic/sub09_Pink Noise Segs_Music Segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic/sub09_Pink Noise Segs_Music Segs_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic/sub10_Pink Noise Segs_Music Segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic/sub10_Pink N

In [54]:
bands = ["Exponent", "Offset"]  # Aperiodic parameters

save_path1 = f"{plots_path}/fdr_source/music_pink_segs/aperiodic_color/"
save_path2 = f"{plots_path}/fdr_source/music_pink_segs/aperiodic_color/merged/"


plot_fooof_relative_difference_results(
    eeg_fooof_music_pink_segs_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="pink_segs", condition2="music_segs",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2, out_filename="src_aperiodic_reldiff_musicsegs_pinksegs.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic_color/sub07_pink_segs_music_segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic_color/sub07_pink_segs_music_segs_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic_color/sub09_pink_segs_music_segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic_color/sub09_pink_segs_music_segs_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic_color/sub10_pink_segs_music_segs_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/music_pink_segs/aperiodic_color/sub10_

#### 3. Rest vs. Pink Noise

In [55]:
eeg_fooof_all_subs["sub07"].keys()

dict_keys(['music_segs', 'pink_segs', 'rest_end', 'rest_init', 'rest_mid', 'reward'])

In [56]:
eeg_fooof_rest_pink = {}

eeg_fooof_rest_pink["sub09"] = {} 
eeg_fooof_rest_pink["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_pink["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()

eeg_fooof_rest_pink["sub10"] = {} 
eeg_fooof_rest_pink["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_pink["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()

eeg_fooof_rest_pink["sub12"] = {}
eeg_fooof_rest_pink["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_pink["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()

eeg_fooof_rest_pink_matched = get_matched_legths(eeg_fooof_rest_pink)

Subject: sub09, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 175 → 101 epochs (dropped last 74)
(90, 223, 101) (90, 223, 101)
Subject: sub10, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 95 → 71 epochs (dropped last 24)
(90, 223, 71) (90, 223, 71)
Subject: sub12, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 139 → 115 epochs (dropped last 24)
(90, 223, 115) (90, 223, 115)


In [57]:
sub_id = "sub10"

freqs = psds_all[sub_id]["rest_init"][1]
psds = psds_all[sub_id]["rest_init"][0]

n_regions = 90

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}

_, rest_pink_fooof_bands_pvalues_abs, _ = get_channel_wise_fooof(freqs, eeg_fooof_rest_pink_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_rest_pink = get_fdr_p_values(rest_pink_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)


sub09
Paradigm 0 rest_init
Paradigm 1 pink_noise
sub10
Paradigm 0 rest_init
Paradigm 1 pink_noise
sub12
Paradigm 0 rest_init
Paradigm 1 pink_noise


In [58]:
save_path1 = f"{plots_path}/fdr_source/rest_pink_noise/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/rest_pink_noise/fdr_plots/merged/"

parad1 = "Rest"
parad2 = "Pink Noise"
view = "lateral"
pvals = True 

cmap_color = ListedColormap(["#EDF4F6", "#6FB46F"])   # non-sig pale / SIGNIFICANT green

legend_label1 = "Rest"
legend_label2 = "Pink Noise Cont."
plot_fooof_fdr_results(fdr_corrected_p_vals_rest_pink, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, pat_state_dict, save_path1, 
                       save_path2, out_filename="src_sig_rest_pinkcont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/fdr_plots/sub09_Rest_Pink Noise_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/fdr_plots/sub10_Rest_Pink Noise_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universi

In [59]:
view = "lateral"
pvals = True 

_, eeg_fooof_bands_abs_rest_pink = get_fooof_bands(eeg_fooof_rest_pink_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_rest_pink,
    bands, aal_img, label_region_mapping,
    condition1="rest_init", condition2="pink_noise",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/rest_pink_noise/color/",
    save_dir2=f"{plots_path}/fdr_source/rest_pink_noise/color/merged/", out_filename="src_reldiff_rest_pinkcont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/color/sub09_rest_init_pink_noise_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/color/sub10_rest_init_pink_noise_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Un

In [ ]:
# _, eeg_fooof_bands_abs_rest_pink = get_fooof_bands(eeg_fooof_rest_pink_matched, freqs, psds, bands)

# summary = build_cluster_summary_all(
#     fdr_corrected_p_vals_rest_pink, eeg_fooof_bands_abs_rest_pink,
#     condition1="pink_segs", condition2="music_segs", bands=bands,
#     label_region_mapping=label_region_mapping,
#     region_to_anatomical_lobe=region_to_anatomical_lobe,
# )

# plot_cluster_bubble_heatmap(
#     summary, bands,
#     cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
#     pat_state=pat_state_dict, sub_labels=sub_label_dict,
#     condition1_label="Pink Noise Cont.", condition2_label="Music Cont.",
#     save_path=f"{plots_path}/fdr_source/rest_pink_noise/color/bubble/cluster_bubble_rest_music.pdf",
# )


KeyError: 'pink_segs'

Aperiodic

In [60]:
eeg_fooof_rest_pink_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_pink_matched)


fdr_corrected_p_vals_rest_pink_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_pink_aperiodic, pat_state, n_channels=n_regions)



aperiodic_params = ["Exponent", "Offset"] 

save_path1 = f"{plots_path}/fdr_source/rest_pink_noise/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/rest_pink_noise/aperiodic/merged/"

parad1 = "Rest"
parad2 = "Pink Noise"
view = "lateral"
pvals = True 

cmap_color = ListedColormap(["#EDF4F6", "#6FB46F"])   # non-sig pale / SIGNIFICANT green

plot_fooof_fdr_results(fdr_corrected_p_vals_rest_pink_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, 
                       view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_rest_pinkcont.pdf")  




Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic/sub09_Rest_Pink Noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic/sub09_Rest_Pink Noise_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic/sub10_Rest_Pink Noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic/sub10_Rest_Pink Noise_Offset.pdf
Saved: D:/Taran/OneDrive - R

In [61]:
bands = ["Exponent", "Offset"]  # Aperiodic parameters

save_path1 = f"{plots_path}/fdr_source/rest_pink_noise/aperiodic_color/"
save_path2 = f"{plots_path}/fdr_source/rest_pink_noise/aperiodic_color/merged/"


plot_fooof_relative_difference_results(
    eeg_fooof_rest_pink_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="rest_init", condition2="pink_noise",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2,out_filename="src_aperiodic_reldiff_rest_pinkcont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic_color/sub09_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic_color/sub09_rest_init_pink_noise_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic_color/sub10_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic_color/sub10_rest_init_pink_noise_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic_color/sub12_rest_init_pink_noise_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_pink_noise/aperiodic_color/sub12_

#### 4. Reward vs. Pink Noise

In [62]:
eeg_fooof_pink_noise_reward = {}

eeg_fooof_pink_noise_reward["sub09"] = {} 
eeg_fooof_pink_noise_reward["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub10"] = {} 
eeg_fooof_pink_noise_reward["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub12"] = {}
eeg_fooof_pink_noise_reward["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

eeg_fooof_pink_noise_reward_matched = get_matched_legths(eeg_fooof_pink_noise_reward)

Subject: sub09, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 133 → 101 epochs (dropped last 32)
(90, 223, 101) (90, 223, 101)
Subject: sub10, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 85 → 71 epochs (dropped last 14)
(90, 223, 71) (90, 223, 71)
Subject: sub12, Paradigms: ['pink_noise', 'reward']
Truncated array1 from 115 → 105 epochs (dropped last 10)
(90, 223, 105) (90, 223, 105)


In [63]:
sub_id = "sub10"

freqs = psds_all[sub_id]["rest_init"][1]
psds = psds_all[sub_id]["rest_init"][0]

n_regions = 90

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}


_, pink_noise_reward_fooof_bands_pvalues_abs, fooof_pink_noise_reward = get_channel_wise_fooof(freqs, eeg_fooof_pink_noise_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_pink_noise_reward = get_fdr_p_values(pink_noise_reward_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)

sub09
Paradigm 0 pink_noise
Paradigm 1 reward
sub10
Paradigm 0 pink_noise
Paradigm 1 reward
sub12
Paradigm 0 pink_noise
Paradigm 1 reward


In [64]:
mcs_minus_subjects = ["sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_pink_noise_reward,
    mcs_minus_subjects,
    condition1="pink_noise",
    condition2="reward",   # confirm this matches your actual music-condition key
    bands=bands,
    abs_rel=0,
    n_boot=1000,
    n_channels=90
)

for band in bands:
    print(f"{band}: {sig_mask[band].sum()} / 128 channels significant at group level")


latex_code_2 = latex_group_vs_individual_table(
    fdr_corrected_p_vals_pink_noise_reward, mcs_minus_subjects, sig_mask, bands,
    comparison_label="Continuous Music vs. Continuous Pink Noise",
    mode='append',
    n_channels=90
)

print(latex_code_2)




delta: 0 / 128 channels significant at group level
theta: 0 / 128 channels significant at group level
alpha: 0 / 128 channels significant at group level
beta: 0 / 128 channels significant at group level
gamma low: 0 / 128 channels significant at group level
\midrule

\multirow{5}{*}{\shortstack[l]{Continuous Music vs.\\Continuous Pink Noise}}
& $\delta$ & 0/90 & 0/90 & 0/90 ($\geq$2/3) & 0/90 \\
& $\theta$ & 0/90 & 1/90 & 0/90 ($\geq$2/3) & 0/90 \\
& $\alpha$ & 0/90 & 0/90 & 0/90 ($\geq$2/3) & 0/90 \\
& $\beta$ & 0/90 & 0/90 & 0/90 ($\geq$2/3) & 0/90 \\
& $\gamma$ & 0/90 & 0/90 & 0/90 ($\geq$2/3) & 0/90 \\




In [65]:
save_path1 = f"{plots_path}/fdr_source/pink_noise_reward/fdr_plots/"
save_path2 = f"{plots_path}/fdr_source/pink_noise_reward/fdr_plots/merged/"

parad1 = "Pink Noise"
parad2 = "Music Cont."
view = "lateral"
pvals = True

legend_label1 = "Pink Noise Cont."
legend_label2 = "Music Cont."
cmap_color = ListedColormap(["#EDF4F6", "#6FB46F"])   # non-sig pale / SIGNIFICANT green

plot_fooof_fdr_results(fdr_corrected_p_vals_pink_noise_reward, bands, aal_img, label_region_mapping, parad1, parad2, pvals, cmap_color, view, 
                       pat_state_dict, save_path1, save_path2, out_filename="src_sig_pinkcont_musiccont.pdf")  


Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/fdr_plots/sub09_Pink Noise_Music Cont._gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/fdr_plots/sub10_Pink Noise_Music Cont.

In [66]:

parad1 = "Pink Noise Cont."
parad2 = "Music Cont."
view = "lateral"

_, eeg_fooof_bands_abs_rest_pink = get_fooof_bands(eeg_fooof_pink_noise_reward_matched, freqs, psds, bands)


plot_fooof_relative_difference_results(
    eeg_fooof_bands_abs_rest_pink,
    bands, aal_img, label_region_mapping,
    condition1="pink_noise", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=f"{plots_path}/fdr_source/pink_noise_reward/color/",
    save_dir2=f"{plots_path}/fdr_source/pink_noise_reward/color/merged/", out_filename="src_reldiff_pinkcont_musiccont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/color/sub09_pink_noise_reward_gamma low.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/color/sub10_pink_noise_reward_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universi

In [67]:
_, eeg_fooof_bands_abs_pink_reward = get_fooof_bands(eeg_fooof_pink_noise_reward_matched, freqs, psds, bands)


summary = build_cluster_summary_all(
    fdr_corrected_p_vals_pink_noise_reward, eeg_fooof_bands_abs_pink_reward,
    condition1="pink_noise", condition2="reward", bands=bands,
    label_region_mapping=label_region_mapping,
    region_to_anatomical_lobe=region_to_anatomical_lobe,
)

plot_cluster_bubble_heatmap(
    summary, bands,
    cluster_order=["Frontal", "Parietal", "Temporal", "Occipital", "Insula", "Cingulate", "Limbic", "SCGM"],
    pat_state=pat_state_dict, sub_labels=sub_label_dict,
    condition1_label="Pink Noise Cont.", condition2_label="Music Cont.",
    save_path=f"{plots_path}/fdr_source/pink_noise_reward/color/bubble/src_bubble_pinkcont_musiccont.pdf",
)


Aperiodic

In [68]:
eeg_fooof_pink_noise_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_pink_noise_reward_matched)


fdr_corrected_p_vals_pink_noise_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_pink_noise_reward_aperiodic, pat_state, n_channels=n_regions)


aperiodic_params = ["Exponent", "Offset"] 

save_path1 = f"{plots_path}/fdr_source/pink_noise_reward/aperiodic/"
save_path2 = f"{plots_path}/fdr_source/pink_noise_reward/aperiodic/merged/"

parad1 = "Pink Noise Cont."
parad2 = "Music Cont."
view = "lateral"
pvals = True 

cmap_color = ListedColormap(["#EDF4F6", "#6FB46F"])   # non-sig pale / SIGNIFICANT green

plot_fooof_fdr_results(fdr_corrected_p_vals_pink_noise_reward_aperiodic, aperiodic_params, aal_img, label_region_mapping, parad1, parad2, pvals, 
                       cmap_color, view, pat_state_dict, save_path1, save_path2, out_filename="src_aperiodic_sig_pinkcont_musiccont.pdf")  



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic/sub09_Pink Noise Cont._Music Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic/sub09_Pink Noise Cont._Music Cont._Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic/sub10_Pink Noise Cont._Music Cont._Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic/sub10_Pink Noise Cont._Music Con

In [69]:
save_path1 = f"{plots_path}/fdr_source/pink_noise_reward/aperiodic_color/"
save_path2 = f"{plots_path}/fdr_source/pink_noise_reward/aperiodic_color/merged/"

bands = ["Exponent", "Offset"]  # Aperiodic parameters
 
parad1 = "Pink Noise Cont."
parad2 = "Music Cont."
view = "lateral"




plot_fooof_relative_difference_results(
    eeg_fooof_pink_noise_reward_aperiodic,
    bands, aal_img, label_region_mapping,
    condition1="pink_noise", condition2="reward",
    cmap_color=soft_div, view="lateral",
    pat_state_list=pat_state_dict,
    save_dir1=save_path1,
    save_dir2=save_path2, out_filename="src_aperiodic_reldiff_pinkcont_musiccont.pdf")

Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic_color/sub09_pink_noise_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic_color/sub09_pink_noise_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic_color/sub10_pink_noise_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic_color/sub10_pink_noise_reward_Offset.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic_color/sub12_pink_noise_reward_Exponent.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/aperiodic_color/sub12_pin

In [70]:
def build_vector_grid_pdf_dual_folder(
    image_dir_a: str,
    image_dir_b: str,
    bands: list = None,
    pat_state: list = None,
    save_dir: str = None,
    sub_labels: dict = None,
    cell_width_in: float = 4.0,
    cell_height_in: float = 1.8,
    row_label_width_in: float = 0.7,
    col_label_height_in: float = 0.5,
    col_label_fontsize: int = 18,
    gap_x_in: float = 0.05,
    gap_y_in: float = 0.06,          # gap BETWEEN different bands
    within_gap_y_in: float = 0.01,   # gap between the two stacked panels of the SAME band
    row_label_fontsize: int = 24,
    row_label_vshift_in: float = 0.08,
    out_filename: str = "all_subs_all_bands_dual.pdf",
    divider_color: tuple = (0.5, 0.5, 0.5),  # medium-dark gray, RGB 0-1
    divider_width: float = 0.75,
):
    """
    Like build_vector_grid_pdf, but pulls one panel per (sub, band) from EACH
    of two folders and stacks them vertically (folder_a on top, folder_b
    directly below), for every band. The band's row label is vertically
    centered across BOTH stacked panels.

    Filenames in both folders are matched generically as:
        {sub}_<anything>_{band}.pdf
    i.e. only the first token (sub) and the last underscore-delimited token
    before .pdf (band) are used — whatever paradigm text sits in between is
    ignored.
    """
    import re
    import os
    from pathlib import Path
    import matplotlib
    import fitz  # PyMuPDF

    def _vcenter_rect(width_pt, height_pt, y0, text, fontsize, fontfile, fontname):
        tmp_doc = fitz.open()
        tmp_page = tmp_doc.new_page(width=width_pt, height=height_pt)
        rc = tmp_page.insert_textbox(
            fitz.Rect(0, 0, width_pt, height_pt),
            text, fontsize=fontsize, fontfile=fontfile, fontname=fontname,
            align=fitz.TEXT_ALIGN_CENTER,
        )
        tmp_doc.close()
        shift = max(rc, 0) / 2
        return fitz.Rect(0, y0 + shift, width_pt, y0 + height_pt + shift)

    def _build_file_index(folder):
        pattern = re.compile(r"^(?P<sub>[^_]+)_.*_(?P<band>[^_]+)\.pdf$")
        idx = {}
        for fpath in sorted(Path(folder).glob("*.pdf")):
            m = pattern.match(fpath.name)
            if m is None:
                continue
            idx.setdefault(m.group("band"), {})[m.group("sub")] = fpath
        return idx

    PT = 72

    greek_symbols = {
        'delta': '\u03b4', 'theta': '\u03b8', 'alpha': '\u03b1',
        'beta': '\u03b2', 'gamma': '\u03b3', 'gamma low': '\u03b3',
        'Offset': 'b', 'Exponent': '\u03c7',
    }

    file_index_a = _build_file_index(image_dir_a)
    file_index_b = _build_file_index(image_dir_b)

    if not file_index_a and not file_index_b:
        print(f"No matching files found in {image_dir_a} or {image_dir_b}")
        return

    all_bands_found = set(file_index_a.keys()) | set(file_index_b.keys())
    detected_bands = [b for b in bands if b in all_bands_found] if bands else sorted(all_bands_found)

    all_subs = sorted(
        {s for d in file_index_a.values() for s in d.keys()} |
        {s for d in file_index_b.values() for s in d.keys()}
    )
    n_bands, n_subs = len(detected_bands), len(all_subs)

    cell_w, cell_h = cell_width_in * PT, cell_height_in * PT
    row_lbl_w, col_lbl_h = row_label_width_in * PT, col_label_height_in * PT
    gap_x, gap_y, within_gap = gap_x_in * PT, gap_y_in * PT, within_gap_y_in * PT

    block_h = 2 * cell_h + within_gap   # height of ONE band's stacked pair

    total_w = row_lbl_w + n_subs * (cell_w + gap_x) - gap_x
    total_h = col_lbl_h + n_bands * block_h + (n_bands - 1) * gap_y

    out_doc = fitz.open()
    out_page = out_doc.new_page(width=total_w, height=total_h)

    font_path_bold = os.path.join(matplotlib.get_data_path(), "fonts", "ttf", "DejaVuSans-Bold.ttf")

    # ── column headers (subjects) ─────────────────────────────────────────
    col_label_vshift_in = 0.2
    start_offset = list(pat_state.keys()).index(all_subs[0]) if pat_state else 0
    for col, sub in enumerate(all_subs):
        x = row_lbl_w + col * (cell_w + gap_x)
        sub_label = sub_labels[sub] if sub_labels and sub in sub_labels else f"SUB{start_offset + col + 1}"
        state = f" ({pat_state[sub]})" if pat_state and sub in pat_state else ""
        text = f"{sub_label}{state}"
        y0 = col_label_vshift_in * PT
        out_page.insert_textbox(
            fitz.Rect(x, y0, x + cell_w, col_lbl_h + y0),
            text, fontsize=col_label_fontsize, fontfile=font_path_bold, fontname="F-bold",
            align=fitz.TEXT_ALIGN_CENTER,
        )

    # ── row labels (bands), centered across BOTH stacked panels ────────────
    for row, band in enumerate(detected_bands):
        y_block = col_lbl_h + row * (block_h + gap_y)
        label = greek_symbols.get(band, band.upper())

        row_rect = _vcenter_rect(
            row_lbl_w, block_h, y_block + row_label_vshift_in * PT,
            label, row_label_fontsize, font_path_bold, "F-bold",
        )
        out_page.insert_textbox(
            row_rect, label, fontsize=row_label_fontsize,
            fontfile=font_path_bold, fontname="F-bold",
            align=fitz.TEXT_ALIGN_CENTER,
        )

    # ── panels: folder_a on top, folder_b directly below, per (band, sub) ──
    src_docs = []
    for row, band in enumerate(detected_bands):
        y_block = col_lbl_h + row * (block_h + gap_y)
        y_a = y_block
        y_b = y_a + cell_h + within_gap

        for col, sub in enumerate(all_subs):
            x = row_lbl_w + col * (cell_w + gap_x)

            for y_pos, index, vanchor in ((y_a, file_index_a, "bottom"), (y_b, file_index_b, "top")):
                fpath = index.get(band, {}).get(sub)
                cell_rect = fitz.Rect(x, y_pos, x + cell_w, y_pos + cell_h)

                if fpath is None:
                    out_page.insert_textbox(cell_rect, "N/A", fontsize=12,
                                             align=fitz.TEXT_ALIGN_CENTER, color=(0.5, 0.5, 0.5))
                    continue

                src_doc = fitz.open(str(fpath))
                src_docs.append(src_doc)
                src_page = src_doc[0]

                scale = min(cell_w / src_page.rect.width, cell_h / src_page.rect.height)
                fit_w, fit_h = src_page.rect.width * scale, src_page.rect.height * scale
                offset_x = x + (cell_w - fit_w) / 2
                if vanchor == "bottom":
                    offset_y = y_pos + (cell_h - fit_h)   # push to bottom of its cell
                else:
                    offset_y = y_pos                       # push to top of its cell
                fit_rect = fitz.Rect(offset_x, offset_y, offset_x + fit_w, offset_y + fit_h)

                out_page.show_pdf_page(fit_rect, src_doc, 0)

    # ── solid divider lines between band blocks ─────────────────────────────
    body_left = 0
    body_right = total_w

    for row in range(1, n_bands):
        y_gap_mid = col_lbl_h + row * (block_h + gap_y) - gap_y / 2
        out_page.draw_line(
            fitz.Point(body_left, y_gap_mid),
            fitz.Point(body_right, y_gap_mid),
            color=divider_color, width=divider_width,
        )

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out_path = os.path.join(save_dir, out_filename)
        out_doc.save(out_path)
        print(f"Saved (fully vector) → {out_path}")

    out_doc.close()
    for d in src_docs:
        d.close()

In [71]:
folder_a = plots_path+ "/fdr_source/rest_reward/color/"
folder_b = plots_path + "/fdr_source/rest_reward/fdr_plots/"
outpur_folder = plots_path + "/fdr_source/rest_reward/combined_dual_grid/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_rest_musiccont.pdf",
    within_gap_y_in=0.0,   # tighter spacing between the two same-band panels

)

Saved (fully vector) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/rest_reward/combined_dual_grid/src_dual_rest_musiccont.pdf


In [72]:
folder_a = plots_path+ "/fdr_source/pink_noise_reward/color/"
folder_b = plots_path + "/fdr_source/pink_noise_reward/fdr_plots/"
outpur_folder = plots_path + "/fdr_source/pink_noise_reward/combined_dual_grid/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_pinkcont_musiccont.pdf",
    within_gap_y_in=0.0,   # tighter spacing between the two same-band panels

)

Saved (fully vector) → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots//fdr_source/pink_noise_reward/combined_dual_grid/src_dual_pinkcont_musiccont.pdf


In [73]:
folder_a = path_analysis_data_new + "analysis_plots/fdr_source/music_pink_segs/color/"
folder_b = path_analysis_data_new + "analysis_plots/fdr_source/music_pink_segs/fdr_plots/"
outpur_folder = path_analysis_data_new + "analysis_plots/fdr_source/music_pink_segs/combined_dual_grid/"

build_vector_grid_pdf_dual_folder(
    image_dir_a=folder_a,
    image_dir_b=folder_b,
    bands=["delta", "theta", "alpha", "beta", "gamma low"],
    pat_state=pat_state_dict,
    save_dir=outpur_folder,
    out_filename="src_dual_musicsegs_pinksegs.pdf",
    within_gap_y_in=0.0,   # tighter spacing between the two same-band panels

)

Saved (fully vector) → D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/analysis_plots/fdr_source/music_pink_segs/combined_dual_grid/src_dual_musicsegs_pinksegs.pdf


In [ ]:
import fitz
for path in [
    "music_segs_pink_segs_abs_fdr.pdf",
    "music_segs_pink_segs_abs.pdf",
    "music_segs_pink_segs_abs_aperiodic_fdr.pdf",
    "music_segs_pink_segs_abs_aperiodic.pdf",
]:
    doc = fitz.open(path)
    r = doc[0].rect
    print(f"{path}: {r.width:.1f} x {r.height:.1f} pt, aspect ratio = {r.width/r.height:.3f}")
    doc.close()